<!-- sync check -->

## Setup

In [0]:
%pip install -r ../requirements.txt
%pip install -r ../requirements-optional.txt
dbutils.library.restartPython()

## 1. Project root on sys.path
(required so from src.etl import ... resolves — Databricks doesn't put the notebook's parent folder on sys.path automatically)

In [0]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"project_root: {project_root}")

## 2. Imports

In [0]:
import time
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, recall_score

from src.etl import (
    load_raw_data, drop_high_missing_columns, add_missingness_flags,
    impute_missing, treat_outliers, preserve_email_raw, encode_categoricals,
)
from src.feature_engineering import load_clean_data, engineer_features, select_features
from src.train import load_data, train, evaluate as evaluate_holdout, BEST_XGB_PARAMS
from src.evaluate import (
    load_test_split, error_breakdown, tune_threshold,
    MIN_PR_AUC, MIN_RECALL_AT_DEFAULT_THRESHOLD,
)
from src.explain import load_test_sample

## 3. Config

In [0]:
RAW_DIR = "/Volumes/workspace/default/fraud_detection_data"
PROCESSED_DIR = "/Volumes/workspace/default/fraud_detection_data/processed"
MODELS_DIR = "/Volumes/workspace/default/fraud_detection_data/models"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"RAW_DIR:       {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")
print(f"MODELS_DIR:    {MODELS_DIR}")

## 4. ETL — raw data → df_clean.csv

In [0]:
print("=" * 60); print("1. LOAD RAW DATA"); print("=" * 60)
df_raw = load_raw_data(RAW_DIR)

print("\n" + "=" * 60); print("2. MISSING VALUE HANDLING"); print("=" * 60)
df_clean, drop_cols = drop_high_missing_columns(df_raw)
df_clean, useful_flags, weak_flags = add_missingness_flags(df_clean)
df_clean, medians, modes = impute_missing(df_clean)

print("\n" + "=" * 60); print("3. OUTLIER TREATMENT"); print("=" * 60)
df_clean, outlier_bounds = treat_outliers(df_clean)

print("\n" + "=" * 60); print("4. ENCODE CATEGORICALS"); print("=" * 60)
df_clean = preserve_email_raw(df_clean)
df_clean, label_encoders = encode_categoricals(df_clean)

clean_path = f"{PROCESSED_DIR}/df_clean.csv"
df_clean.to_csv(clean_path, index=False)
print(f"\n✅ df_clean.csv saved: {df_clean.shape} -> {clean_path}")

etl_artifacts = {
    "drop_cols": drop_cols,
    "useful_flags": useful_flags,
    "weak_flags": weak_flags,
    "medians": medians,
    "modes": modes,
    "outlier_bounds": outlier_bounds,
    "label_encoders": label_encoders,
}
joblib.dump(etl_artifacts, f"{PROCESSED_DIR}/etl_artifacts.pkl")
print(f"✅ etl_artifacts.pkl saved -> {PROCESSED_DIR}/etl_artifacts.pkl")

## 5. Feature Engineering — df_clean.csv → df_final.csv

In [0]:
print("=" * 60); print("FEATURE ENGINEERING"); print("=" * 60)
df_clean_reloaded = load_clean_data(clean_path)
df_clean_reloaded, feature_artifacts = engineer_features(df_clean_reloaded)
df_final, top_features = select_features(df_clean_reloaded)

final_path = f"{PROCESSED_DIR}/df_final.csv"
df_final.to_csv(final_path, index=False)
print(f"\n✅ df_final.csv saved: {df_final.shape} -> {final_path}")

feat_artifacts = {
    "card_avg_lookup": feature_artifacts["card_avg_lookup"],
    "top_features": top_features,
}
joblib.dump(feat_artifacts, f"{PROCESSED_DIR}/feature_artifacts.pkl")
print(f"✅ feature_artifacts.pkl saved -> {PROCESSED_DIR}/feature_artifacts.pkl")

## 6. Train — df_final.csv → tuned XGBoost model

In [0]:
print("=" * 60); print("LOADING DATA"); print("=" * 60)
X, y = load_data(final_path)
print(f"Loaded {X.shape[0]:,} rows, {X.shape[1]} features")
print(f"Fraud rate: {y.mean() * 100:.2f}%")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
scaler.fit(X_train)  # fit on train only — kept for parity with LR/KNN/ANN, unused by XGBoost itself

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

print("\nUsing saved best hyperparameters:")
for k, v in BEST_XGB_PARAMS.items():
    print(f"  {k}: {v}")

print("\n" + "=" * 60); print("TRAINING"); print("=" * 60)
start = time.time()
model = train(X_train, y_train, BEST_XGB_PARAMS, scale_pos_weight)
train_time = time.time() - start
print(f"Training complete in {train_time:.1f}s")

print("\n" + "=" * 60); print("HOLD-OUT EVALUATION"); print("=" * 60)
metrics = evaluate_holdout(model, X_test, y_test, threshold=0.75)
metrics["train_time_seconds"] = round(train_time, 1)
metrics["n_train_rows"] = int(len(X_train))
metrics["n_test_rows"] = int(len(X_test))
metrics["hyperparameters"] = BEST_XGB_PARAMS
for k, v in metrics.items():
    print(f"  {k}: {v}")

joblib.dump(model, f"{MODELS_DIR}/model.pkl")
joblib.dump(scaler, f"{MODELS_DIR}/scaler.pkl")
with open(f"{MODELS_DIR}/feature_names.json", "w") as f:
    json.dump(list(X.columns), f, indent=2)
with open(f"{MODELS_DIR}/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=float)

print(f"\n✅ Artifacts saved to {MODELS_DIR}/")
print("   model.pkl, scaler.pkl, feature_names.json, metrics.json")

## 7. Evaluate — quality gate

In [0]:
print("=" * 60); print("LOADING MODEL + TEST SPLIT"); print("=" * 60)
eval_model = joblib.load(f"{MODELS_DIR}/model.pkl")
X_test_eval, y_test_eval = load_test_split(final_path)
print(f"Test set: {X_test_eval.shape}")

y_proba = eval_model.predict_proba(X_test_eval)[:, 1]
y_pred_default = (y_proba >= 0.5).astype(int)

print("\n" + "=" * 60); print("ERROR ANALYSIS (threshold=0.5)"); print("=" * 60)
errors = error_breakdown(y_test_eval.values, y_pred_default)
for k, v in errors.items():
    print(f"  {k}: {v:,}")

print("\n" + "=" * 60); print("THRESHOLD TUNING"); print("=" * 60)
threshold_results = tune_threshold(y_test_eval.values, y_proba)
best = threshold_results["best"]
print(f"Best threshold by F1: {best['threshold']}")
print(f"  Precision: {best['precision']:.4f}")
print(f"  Recall:    {best['recall']:.4f}")
print(f"  F1:        {best['f1']:.4f}")

print("\n" + "=" * 60); print("QUALITY GATE"); print("=" * 60)
pr_auc = average_precision_score(y_test_eval, y_proba)
recall_at_default = recall_score(y_test_eval, y_pred_default)

gate_passed = pr_auc >= MIN_PR_AUC and recall_at_default >= MIN_RECALL_AT_DEFAULT_THRESHOLD
print(f"PR-AUC:                     {pr_auc:.4f}  (minimum required: {MIN_PR_AUC})")
print(f"Recall @ threshold=0.5:     {recall_at_default:.4f}  (minimum required: {MIN_RECALL_AT_DEFAULT_THRESHOLD})")
print(f"\n{'✅ PASSED' if gate_passed else '❌ FAILED'} — model is {'' if gate_passed else 'NOT '}approved for deployment")

evaluation_report = {
    "pr_auc": float(pr_auc),
    "recall_at_0.5": float(recall_at_default),
    "gate_passed": bool(gate_passed),
    "error_breakdown_at_0.5": errors,
    "recommended_threshold": best["threshold"],
    "threshold_scan": threshold_results["scan"],
}
with open(f"{MODELS_DIR}/evaluation_report.json", "w") as f:
    json.dump(evaluation_report, f, indent=2)
print(f"\n✅ evaluation_report.json saved -> {MODELS_DIR}/evaluation_report.json")

## 8. Explain — SHAP

In [0]:
import shap

print("=" * 60); print("LOADING MODEL + TEST SAMPLE"); print("=" * 60)
explain_model_obj = joblib.load(f"{MODELS_DIR}/model.pkl")
X_sample, y_sample = load_test_sample(final_path, sample_size=2000)
print(f"SHAP sample: {X_sample.shape}")

print("Computing SHAP values (this takes 1-2 minutes)...")
explainer = shap.TreeExplainer(explain_model_obj)
shap_values = explainer.shap_values(X_sample)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance = pd.Series(mean_abs_shap, index=X_sample.columns).sort_values(ascending=False)

print("\nTop 15 features by mean |SHAP value|:")
print(importance.head(15).to_string())

fraud_positions = X_sample[y_sample == 1].index.tolist()
example_case = None
if fraud_positions:
    idx = fraud_positions[0]
    predicted_prob = explain_model_obj.predict_proba(X_sample.iloc[[idx]])[0, 1]
    example_case = {
        "row_index": int(idx),
        "predicted_probability": float(predicted_prob),
        "top_shap_contributions": pd.Series(
            shap_values[idx], index=X_sample.columns
        ).abs().sort_values(ascending=False).head(10).to_dict(),
    }
    print(f"\nExample fraud case (row {idx}): predicted probability {predicted_prob:.4f}")

shap_report = {
    "n_samples": len(X_sample),
    "base_value": float(explainer.expected_value),
    "top_features": importance.head(20).to_dict(),
    "example_fraud_case": example_case,
}
with open(f"{MODELS_DIR}/shap_report.json", "w") as f:
    json.dump(shap_report, f, indent=2)
print(f"\n✅ shap_report.json saved -> {MODELS_DIR}/shap_report.json")

## 9. Summary

In [0]:
print("=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)
print(f"Processed data:  {PROCESSED_DIR}")
print(f"Model artifacts: {MODELS_DIR}")
print(f"Test PR-AUC:     {pr_auc:.4f}")
print(f"Quality gate:    {'PASSED' if gate_passed else 'FAILED'}")